In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [3]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((.5,.5,.5),(.5,.5,.5))
])
trainset=CIFAR10(root="./data",train=True, download=True ,transform=transform)
testset=CIFAR10(root="./data",train=False, download=True ,transform=transform)

100.0%
C:\Users\Ritvika Jain\ai\venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainLoader=DataLoader(trainset,batch_size=64,shuffle=True)
testLoader=DataLoader(testset,batch_size=64)

In [11]:
# build the CNN

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
            
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [12]:
model=CNN()

In [13]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [19]:
# training CNN

epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    model.train()

    for images, labels in trainLoader:
        outputs=model.forward(images) 
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()

    # epoch_training_loss = running_loss / len(trainLoader)

    print(f"epoch={epoch} & loss ={epoch_training_loss/len(trainLoader)}")

epoch=0 & loss =2.3308707030532916
epoch=1 & loss =15.317163385393675
epoch=2 & loss =4.163886584589243
epoch=3 & loss =2.801669409207981
epoch=4 & loss =347.80725209578833
epoch=5 & loss =2.370614770123416
epoch=6 & loss =2.3524026068885004
epoch=7 & loss =2.3723509796440143
epoch=8 & loss =2.399044276503346
epoch=9 & loss =2.3813512233821936


In [23]:
# evaluation

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testLoader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Correct Predictions : {correct_labels}")
print(f"Total Samples       : {total_labels}")
print(f"Accuracy            : {(correct_labels / total_labels) * 100:.2f}%")

Correct Predictions : 999
Total Samples       : 10000
Accuracy            : 9.99%
